In [4]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
num_servers = 5000

server_ids = [f"VM-{i:04d}" for i in range(1, num_servers + 1)]
departments = ["R&D", "Finance", "HR", "Marketing", "Production"]
regions = ["us-east-1", "ap-south-1", "eu-west-1", "us-west-2"]
instance_types = ["t3.micro", "m5.large", "c5.xlarge", "r5.2xlarge"]
cost_mapping = {"t3.micro": 0.0104, "m5.large": 0.096, "c5.xlarge": 0.170, "r5.2xlarge": 0.504}

server_pool = []
for s_id in server_ids:
    dept = np.random.choice(departments)
    region = np.random.choice(regions)
    itype = np.random.choice(instance_types)
    cost = cost_mapping[itype]

    # Introduce three distinct operational profiles
    profile_type = np.random.choice(["idle", "over_provisioned", "optimized"], p=[0.15, 0.20, 0.65])

    if profile_type == "idle":
        cpu_mean, cpu_std = 2.0, 0.5
        net_mean, net_std = 0.1, 0.05
    elif profile_type == "over_provisioned":
        cpu_mean, cpu_std = 12.0, 2.0
        net_mean, net_std = 5.0, 1.5
    else:
        cpu_mean, cpu_std = 45.0, 10.0
        net_mean, net_std = 50.0, 15.0

    server_pool.append({
        "Resource_ID": s_id,
        "Department": dept,
        "Region": region,
        "Instance_Type": itype,
        "Cost_Per_Hour_USD": cost,
        "CPU_Mean": cpu_mean,
        "CPU_Std": cpu_std,
        "Net_Mean": net_mean,
        "Net_Std": net_std
    })

meta_df = pd.DataFrame(server_pool)

# Generate time-series telemetry data over a rolling 7-day window
base_date = datetime.now() - timedelta(days=7)
date_list = [base_date + timedelta(hours=x) for x in range(168)]

telemetry_records = []
for index, row in meta_df.sample(n=1000, random_state=42).iterrows():
    for dt in date_list:
        cpu = max(0.0, min(100.0, np.random.normal(row["CPU_Mean"], row["CPU_Std"])))
        net = max(0.0, np.random.normal(row["Net_Mean"], row["Net_Std"]))

        telemetry_records.append({
            "Timestamp": dt,
            "Resource_ID": row["Resource_ID"],
            "CPU_Utilization_Percentage": cpu,
            "Network_Traffic_MB": net
        })

telemetry_df = pd.DataFrame(telemetry_records)

meta_df.to_csv("cloud_metadata.csv", index=False)
telemetry_df.to_csv("cloud_telemetry.csv", index=False)

print("Dataset generated successfully!")
print(f"Metadata shape: {meta_df.shape} | Telemetry shape: {telemetry_df.shape}")


Dataset generated successfully!
Metadata shape: (5000, 9) | Telemetry shape: (168000, 4)


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# STEP 2: FEATURE ENGINEERING & DATA MATH
# ==========================================
print("--- Starting Step 2: Feature Engineering ---")
meta_df = pd.read_csv("cloud_metadata.csv")
telemetry_df = pd.read_csv("cloud_telemetry.csv")

# Aggregate time-series telemetry metrics per unique virtual machine
agg_features = telemetry_df.groupby("Resource_ID").agg(
    Avg_CPU=("CPU_Utilization_Percentage", "mean"),
    Max_CPU=("CPU_Utilization_Percentage", "max"),
    Std_CPU=("CPU_Utilization_Percentage", "std"),
    Total_Network_MB=("Network_Traffic_MB", "sum"),
    Total_Active_Hours=("Timestamp", "count")
).reset_index()

# Merge telemetry features with server cost configuration metadata
features_df = pd.merge(agg_features, meta_df, on="Resource_ID")

# Label data into optimization categories based on cloud architecture thresholds
conditions = [
    (features_df["Max_CPU"] < 5.0) & (features_df["Total_Network_MB"] < 50.0),
    (features_df["Avg_CPU"] < 15.0) & (features_df["Max_CPU"] < 40.0) & (features_df["Max_CPU"] >= 5.0),
]
choices = ["Idle", "Over-Provisioned"]
features_df["Optimization_Target"] = np.select(conditions, choices, default="Optimized")

# Calculate wasted dollar spend based on resource state
def calculate_waste(row):
    if row["Optimization_Target"] == "Idle":
        return row["Total_Active_Hours"] * row["Cost_Per_Hour_USD"]
    elif row["Optimization_Target"] == "Over-Provisioned":
        return row["Total_Active_Hours"] * (row["Cost_Per_Hour_USD"] * 0.5)
    return 0.0

features_df["Wasted_Spend_USD"] = features_df.apply(calculate_waste, axis=1)
features_df.to_csv("engineered_cloud_features.csv", index=False)
print("Step 2 Done! Dataset labeled and processed.")

# ==========================================
# STEP 3: TRAIN RANDOM FOREST CLASSIFIER
# ==========================================
print("\n--- Starting Step 3: Training Random Forest ---")

# Define our prediction inputs (features) and targeted output (label)
X = features_df[["Avg_CPU", "Max_CPU", "Std_CPU", "Total_Network_MB"]]
y = features_df["Optimization_Target"]

# Split the data securely: 80% for training the AI, 20% for testing it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Initialize and train our Random Forest model
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Run model predictions on unseen test assets
y_pred = clf.predict(X_test)

# Print a professional evaluation metrics summary report
print(f"Model Training Complete! Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Evaluation Metrics Report:")
print(classification_report(y_test, y_pred))


--- Starting Step 2: Feature Engineering ---
Step 2 Done! Dataset labeled and processed.

--- Starting Step 3: Training Random Forest ---
Model Training Complete! Accuracy Score: 100.00%

Classification Evaluation Metrics Report:
                  precision    recall  f1-score   support

            Idle       1.00      1.00      1.00        30
       Optimized       1.00      1.00      1.00       129
Over-Provisioned       1.00      1.00      1.00        41

        accuracy                           1.00       200
       macro avg       1.00      1.00      1.00       200
    weighted avg       1.00      1.00      1.00       200



In [6]:
import pandas as pd

# Load the features and labels generated in our pipeline
df = pd.read_csv("engineered_cloud_features.csv")

# Calculate global infrastructure cost summaries
total_servers = len(df)
total_wasted_spend = df["Wasted_Spend_USD"].sum()

# Count server breakdown distributions
status_counts = df["Optimization_Target"].value_counts()
idle_count = status_counts.get("Idle", 0)
over_count = status_counts.get("Over-Provisioned", 0)
optimized_count = status_counts.get("Optimized", 0)

# Calculate financial distributions per department
dept_waste = df.groupby("Department")["Wasted_Spend_USD"].sum().reset_index()
worst_dept = dept_waste.sort_values(by="Wasted_Spend_USD", ascending=False).iloc[0]

# Print the executive audit dashboard report layout
print("=" * 60)
print("     GLOBAL CLOUD ARCHITECTURE OPTIMIZATION REPORT     ")
print("=" * 60)
print(f"Total Cloud Server Assets Audited : {total_servers:,} VMs")
print(f"Healthy / Optimized Deployments   : {optimized_count} Servers")
print(f"Actionable Waste Alerts Triggered : {idle_count + over_count} Servers")
print(f"  └─ Absolute Idle Instances     : {idle_count} (Immediate Shutdown)")
print(f"  └─ Over-Provisioned Instances  : {over_count} (Immediate Downsizing)")
print("-" * 60)
print(" FINANCIAL AUDIT & DIRECT COMMERCIAL IMPACT")
print("-" * 60)
print(f"Total Accumulated Financial Waste : ${total_wasted_spend:,.2f} USD")
print(f"Primary Spending Bottleneck Dept  : Department '{worst_dept['Department']}'")
print(f"  └─ Wasted Department Budget     : ${worst_dept['Wasted_Spend_USD']:,.2f} USD")
print("\n[RECOMMENDATION]: Execute automated cron scripts to isolate and kill")
print("the identified Idle micro-assets to immediately recover losses.")
print("=" * 60)


     GLOBAL CLOUD ARCHITECTURE OPTIMIZATION REPORT     
Total Cloud Server Assets Audited : 1,000 VMs
Healthy / Optimized Deployments   : 643 Servers
Actionable Waste Alerts Triggered : 357 Servers
  └─ Absolute Idle Instances     : 152 (Immediate Shutdown)
  └─ Over-Provisioned Instances  : 205 (Immediate Downsizing)
------------------------------------------------------------
 FINANCIAL AUDIT & DIRECT COMMERCIAL IMPACT
------------------------------------------------------------
Total Accumulated Financial Waste : $8,831.29 USD
Primary Spending Bottleneck Dept  : Department 'Finance'
  └─ Wasted Department Budget     : $2,124.09 USD

[RECOMMENDATION]: Execute automated cron scripts to isolate and kill
the identified Idle micro-assets to immediately recover losses.
